**emsemable程式碼**

In [ ]:
# ============================================================
# VeriPromise ESG M2 - Ensemble + 學姊 Task3（W_avg 優化權重版）
# ============================================================
# 權重來源：second-mix.ipynb 誠實巢狀 CV 調出的 W_avg
# 策略：
#   Task1/2/4 → M2 7模型（W_avg 任務專屬權重）
#   Task3     → 學姊的 probs_ensemble.pkl
# ============================================================

import os, json, pickle, copy, urllib.request, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive 已掛載")
except:
    print("⚠️ 非 Colab 環境")

# ============================================================
# 1. 路徑設定
# ============================================================

PKL_DIR = "/content/drive/MyDrive/ESGtest/M2_pkl"
JIE_PKL = "/content/drive/MyDrive/ESGtest/M2_pkl/0.62esg/probs_ensemble.pkl"
OUT_DIR = "/content/drive/MyDrive/ESGtest"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 2. 任務設定
# ============================================================

EVAL_FIELDS = {
    "promise_status":        ["Yes", "No"],
    "verification_timeline": ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years", "N/A"],
    "evidence_status":       ["Yes", "No", "N/A"],
    "evidence_quality":      ["Clear", "Not Clear", "Misleading", "N/A"],
}
FIELD_WEIGHTS = {
    "promise_status": 0.20, "verification_timeline": 0.15,
    "evidence_status": 0.30, "evidence_quality": 0.35,
}
LABEL2ID   = {f: {l: i for i, l in enumerate(ls)} for f, ls in EVAL_FIELDS.items()}
NUM_LABELS = {f: len(ls) for f, ls in EVAL_FIELDS.items()}

EXPECTED_MODELS = ["electra", "ernie", "large", "lert", "macbert", "mengzi", "roberta"]

# ── W_avg：second-mix.ipynb 誠實巢狀 CV 調出的任務專屬權重 ──
# 各任務對應每個模型的最佳權重
W_AVG = {
    "promise_status": {
        "electra": 1.00, "ernie": 0.40, "large": 1.10,
        "lert":    1.20, "macbert": 0.70, "mengzi": 1.00, "roberta": 0.90,
    },
    "verification_timeline": {
        "electra": 0.20, "ernie": 0.60, "large": 1.70,
        "lert":    0.40, "macbert": 0.00, "mengzi": 0.00, "roberta": 0.60,
    },
    "evidence_status": {
        "electra": 0.10, "ernie": 1.00, "large": 2.00,
        "lert":    1.20, "macbert": 0.50, "mengzi": 0.40, "roberta": 0.70,
    },
    "evidence_quality": {
        "electra": 0.40, "ernie": 1.30, "large": 1.20,
        "lert":    1.00, "macbert": 0.30, "mengzi": 0.80, "roberta": 0.40,
    },
}

# ============================================================
# 3. 資料載入
# ============================================================

DATA_URL = "https://raw.githubusercontent.com/veripromiseesg/veripromiseesgdataset/ac91c1c8b5d116edf6fc44cccc1ee3b618f5a207/vpesg4ktrain1000v1.json"
urllib.request.urlretrieve(DATA_URL, "data.json")
with open("data.json", encoding="utf-8") as f: data = json.load(f)
for item in data:
    if item.get("verification_timeline") == "longer_than_5_years":
        item["verification_timeline"] = "more_than_5_years"

val_data = []
for vpath in ["/content/drive/MyDrive/ESGtest/aicup驗證集/vpesg4k_val_1000.json",
              "/content/drive/MyDrive/ESGtest/vpesg4k_val_1000.json"]:
    try:
        with open(vpath, encoding="utf-8") as f: val_data = json.load(f)
        for item in val_data:
            if item.get("verification_timeline") == "longer_than_5_years":
                item["verification_timeline"] = "more_than_5_years"
        print(f"✅ 驗證集: {len(val_data)} 筆"); break
    except: continue

train_data = list(data) + list(val_data)
print(f"✅ 訓練池: {len(train_data)} 筆")

test_data = []
for tpath in ["/content/drive/MyDrive/ESGtest/vpesg4k_test_2000.json",
              "/content/drive/MyDrive/ESGtest/測試集/vpesg4k_test_2000.json"]:
    try:
        with open(tpath, encoding="utf-8") as f: test_data = json.load(f)
        print(f"✅ 測試集: {len(test_data)} 筆"); break
    except: continue

n_test  = len(test_data)
n_train = len(train_data)

# ============================================================
# 4. 載入 M2 pkl
# ============================================================

print(f"\n{'='*60}\n載入 M2 pkl...")
all_results = {}
for short in EXPECTED_MODELS:
    path = os.path.join(PKL_DIR, f"M2_{short}.pkl")
    if not os.path.exists(path):
        print(f"  ⚠️  {short}: 找不到"); continue
    with open(path, "rb") as f: obj = pickle.load(f)
    ok = "test" in obj and all(
        t in obj["test"] and obj["test"][t].shape[0] == n_test
        for t in EVAL_FIELDS)
    if ok:
        all_results[short] = obj
        print(f"  ✅ {short:10s}: OOF CV={obj.get('cv',0):.5f}")
    else:
        print(f"  ⚠️  {short}: 格式不符")

print(f"共載入 {len(all_results)} 個模型")

# ============================================================
# 5. 載入學姊 pkl
# ============================================================

print(f"\n載入學姊 pkl...")
with open(JIE_PKL, "rb") as f:
    jie_res = pickle.load(f)
print(f"  ✅ 學姊 pkl 載入成功 | tasks: {list(jie_res['test'].keys())}")

# ============================================================
# 6. decode / wmf1
# ============================================================

def decode_probs(probs, n):
    y_idx=LABEL2ID["promise_status"]["Yes"]; n_idx=LABEL2ID["promise_status"]["No"]
    vt_labs=["already","within_2_years","between_2_and_5_years","more_than_5_years"]
    vt_idx=[LABEL2ID["verification_timeline"][l] for l in vt_labs]
    es_labs=["Yes","No"]; es_idx=[LABEL2ID["evidence_status"][l] for l in es_labs]
    eq_labs=["Clear","Not Clear","Misleading"]
    eq_idx=[LABEL2ID["evidence_quality"][l] for l in eq_labs]
    out=[]
    for i in range(n):
        ps_p=probs["promise_status"][i]
        ps="Yes" if ps_p[y_idx]>=ps_p[n_idx] else "No"
        r={"promise_status":ps}
        if ps=="No":
            r.update(verification_timeline="N/A",evidence_status="N/A",evidence_quality="N/A")
        else:
            r["verification_timeline"]=vt_labs[int(np.argmax(probs["verification_timeline"][i,vt_idx]))]
            es=es_labs[int(np.argmax(probs["evidence_status"][i,es_idx]))]
            r["evidence_status"]=es
            r["evidence_quality"]=eq_labs[int(np.argmax(probs["evidence_quality"][i,eq_idx]))] if es=="Yes" else "N/A"
        out.append(r)
    return out

def compute_wmf1(gt, preds):
    s=0.0
    for f,labs in EVAL_FIELDS.items():
        s+=f1_score([g[f] for g in gt],[p[f] for p in preds],
                    labels=labs,average="macro",zero_division=0)*FIELD_WEIGHTS[f]
    return s

# ============================================================
# 7. W_avg 加權 blend（任務專屬權重）
# ============================================================

def blend_wavg(results, split="test"):
    """用 W_avg 對各任務分別加權"""
    probs = {}
    for f in EVAL_FIELDS:
        acc = np.zeros((n_test, NUM_LABELS[f]), dtype=np.float64)
        total_w = 0.0
        for short, obj in results.items():
            w = W_AVG[f].get(short, 1.0)
            if w == 0: continue
            acc += w * obj[split][f]
            total_w += w
        probs[f] = acc / max(total_w, 1e-9)
    return probs

# ============================================================
# 8. OOF CV 確認
# ============================================================

probs_oof = blend_wavg(all_results, split="oof")
oof_cv = compute_wmf1(train_data, decode_probs(probs_oof, n_train))
print(f"\n{'='*60}")
print(f"⭐ W_avg 優化權重 OOF CV: {oof_cv:.5f}")
print(f"   現有 LB (weighted):    0.61860")
print(f"   誠實巢狀 CV (notebook): 0.62360")

# ============================================================
# 9. 儲存函式
# ============================================================

def save_submission(probs, filename, label=""):
    preds = decode_probs(probs, n_test)
    rows  = [{"id": str(test_data[i]["id"]), **preds[i]} for i in range(n_test)]
    df    = pd.DataFrame(rows)[["id","promise_status","verification_timeline",
                                 "evidence_status","evidence_quality"]]
    out   = os.path.join(OUT_DIR, filename)
    df.to_csv(out, index=False, encoding="utf-8-sig", na_rep="N/A")
    no_pct = (df["evidence_status"]=="No").mean()*100
    cls_n  = (df["evidence_quality"]=="Clear").sum()
    nc_n   = (df["evidence_quality"]=="Not Clear").sum()
    mis_n  = (df["evidence_quality"]=="Misleading").sum()
    print(f"\n{'='*60}")
    print(f"✅ {label} → {filename}")
    print(f"  evidence No%:  {no_pct:.1f}%  {'✅' if 13<=no_pct<=14 else '⚠️'}")
    print(f"  quality: Clear={cls_n}  NC={nc_n}  Mis={mis_n}")
    return df, rows

# ============================================================
# 10. 版本 A：W_avg 純 M2（所有任務用優化權重）
# ============================================================

probs_wavg = blend_wavg(all_results, split="test")
df_a, rows_a = save_submission(probs_wavg, "submission_wavg.csv", "W_avg 優化權重")

# ============================================================
# 11. 版本 B：W_avg Task1/2/4 + 學姊 Task3（主要目標）
# ============================================================

probs_b = copy.deepcopy(probs_wavg)
probs_b["evidence_quality"] = jie_res["test"]["evidence_quality"]
df_b, rows_b = save_submission(probs_b, "submission_wavg_jie_t3.csv",
                                "W_avg Task1/2/4 + 學姊 Task3")

# ============================================================
# 12. 版本 C：W_avg Task1/2/4 + Task3 平均（M2+學姊）
# ============================================================

probs_c = copy.deepcopy(probs_wavg)
probs_c["evidence_quality"] = (
    probs_wavg["evidence_quality"] + jie_res["test"]["evidence_quality"]
) / 2
df_c, rows_c = save_submission(probs_c, "submission_wavg_avg_t3.csv",
                                "W_avg Task1/2/4 + Task3 平均")

# ============================================================
# 13. 總結
# ============================================================

print(f"""
{'='*60}
W_avg OOF CV:          {oof_cv:.5f}
誠實巢狀 CV (notebook): 0.62360
現有最佳 LB:            0.61860

提交優先順序：
  ⭐ submission_wavg_jie_t3.csv   ← W_avg + 學姊 Task3（最佳預期）
  ⭐ submission_wavg_avg_t3.csv   ← W_avg + Task3 M2/學姊平均（保守）
     submission_wavg.csv          ← 純 W_avg（無學姊 Task3）

Mis > 0 → 學姊有預測到 Misleading → Task3 更強！
{'='*60}
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive 已掛載
✅ 驗證集: 1000 筆
✅ 訓練池: 2000 筆
✅ 測試集: 2000 筆

載入 M2 pkl...
  ✅ electra   : OOF CV=0.60159
  ✅ ernie     : OOF CV=0.57624
  ✅ large     : OOF CV=0.61926
  ✅ lert      : OOF CV=0.62126
  ✅ macbert   : OOF CV=0.61428
  ✅ mengzi    : OOF CV=0.61283
  ✅ roberta   : OOF CV=0.61494
共載入 7 個模型

載入學姊 pkl...
  ✅ 學姊 pkl 載入成功 | tasks: ['promise_status', 'verification_timeline', 'evidence_status', 'evidence_quality']

⭐ W_avg 優化權重 OOF CV: 0.63588
   現有 LB (weighted):    0.61860
   誠實巢狀 CV (notebook): 0.62360

✅ W_avg 優化權重 → submission_wavg.csv
  evidence No%:  15.3%  ⚠️
  quality: Clear=1157  NC=206  Mis=0

✅ W_avg Task1/2/4 + 學姊 Task3 → submission_wavg_jie_t3.csv
  evidence No%:  15.3%  ⚠️
  quality: Clear=1115  NC=248  Mis=0

✅ W_avg Task1/2/4 + Task3 平均 → submission_wavg_avg_t3.csv
  evidence No%:  15.3%  ⚠️
  quality: Clear=1143  NC=220  Mis=0

W_avg OO

In [ ]:
from sklearn.metrics import f1_score

# ── OOF blend（使用 M2 的 oof probs）──
probs_oof_wavg = blend_wavg(all_results, split="oof")

# 版本 A OOF CV（純 W_avg）
cv_a = compute_wmf1(train_data, decode_probs(probs_oof_wavg, n_train))

# 版本 B OOF CV（W_avg + 學姊 Task3）
probs_oof_b = copy.deepcopy(probs_oof_wavg)
probs_oof_b["evidence_quality"] = jie_res["oof"]["evidence_quality"]
cv_b = compute_wmf1(train_data, decode_probs(probs_oof_b, n_train))

# 版本 C OOF CV（W_avg + Task3 平均）
probs_oof_c = copy.deepcopy(probs_oof_wavg)
probs_oof_c["evidence_quality"] = (
    probs_oof_wavg["evidence_quality"] + jie_res["oof"]["evidence_quality"]
) / 2
cv_c = compute_wmf1(train_data, decode_probs(probs_oof_c, n_train))

print(f"{'='*60}")
print(f"OOF CV 比較：")
print(f"  版本 A（W_avg）:            {cv_a:.5f}")
print(f"  版本 B（W_avg + 學姊 T3）:  {cv_b:.5f}")
print(f"  版本 C（W_avg + T3 平均）:  {cv_c:.5f}")
print(f"  現有 LB:                    0.61860")

OOF CV 比較：
  版本 A（W_avg）:            0.63588
  版本 B（W_avg + 學姊 T3）:  0.63122
  版本 C（W_avg + T3 平均）:  0.63315
  現有 LB:                    0.61860


In [ ]:
for fname, label in [
    ("submission_wavg.csv",         "版本 A（W_avg）"),
    ("submission_wavg_jie_t3.csv",  "版本 B（W_avg + 學姊 T3）"),
    ("submission_wavg_avg_t3.csv",  "版本 C（W_avg + T3 平均）"),
]:
    # ← 加 keep_default_na=False 防止 N/A 被當 NaN
    df = pd.read_csv(
        f"/content/drive/MyDrive/ESGtest/{fname}",
        dtype=str,
        keep_default_na=False
    )
    print(f"\n{'='*50}")
    print(f"{label}")
    print(f"  promise Yes%:  {(df['promise_status']=='Yes').mean()*100:.1f}%")
    print(f"  evidence No%:  {(df['evidence_status']=='No').mean()*100:.1f}%")
    print(f"  quality:", df['evidence_quality'].value_counts().to_dict())


版本 A（W_avg）
  promise Yes%:  83.5%
  evidence No%:  15.3%
  quality: {'Clear': 1157, 'N/A': 637, 'Not Clear': 206}

版本 B（W_avg + 學姊 T3）
  promise Yes%:  83.5%
  evidence No%:  15.3%
  quality: {'Clear': 1115, 'N/A': 637, 'Not Clear': 248}

版本 C（W_avg + T3 平均）
  promise Yes%:  83.5%
  evidence No%:  15.3%
  quality: {'Clear': 1143, 'N/A': 637, 'Not Clear': 220}
